# GAN temporal — previsão antecipada de produtividade com R5 sintético

Este notebook acompanha a pipeline **`2324_temporal_gan_value`**, safra 2023/24,
agora com busca ampliada em **todos os 38 índices/GLCM calculáveis com RRENIR**,
no horizonte **V6 + V8 + V13 + R2 → R5**, com alvo **Produtividade**.
A previsão é feita após R2, usando o histórico real e a projeção sintética de R5.

| Comparação principal, sem dose N (`rs_puro`) | R² OOF |
|---|---:|
| Histórico real — `forecast_history` | 0,683219 |
| Histórico real + R5 sintético — `forecast_fusion` | 0,723257 |
| Ganho da fusão sobre o histórico | **+0,040037** |

**Esses números pertencem à execução anterior, com nove candidatos. A busca ampliada
ainda precisa ser treinada e avaliada; ela não herda esse resultado.**
As células abaixo leem os CSVs da origem selecionada,
recalculam as métricas nas mesmas 24 parcelas e verificam o contraste.
O IC95% bootstrap registrado para o ganho é **[+0,000409; +0,108802]**.
O limite inferior está próximo de zero; a conclusão se restringe a essa avaliação.

**Como usar:** execute as células em ordem. Com `EXECUTAR = False`, padrão,
nenhum preparo ou treino é iniciado; a consulta aos resultados existentes funciona normalmente.
Para retreinar, habilite a execução na configuração e rode as células do protocolo.
Uma nova execução recebe diretório próprio e não substitui os resultados de referência.

As funções canônicas são importadas de `src/milho_experiment`; não há uma segunda
implementação da GAN, da seleção ou dos modelos neste notebook.


## 1. Ambiente, entradas e execução de referência

Use Python ≥ 3.12 e um kernel do ambiente do projeto. Instalação, se necessária, no terminal:

```bash
python -m pip install -e ".[analysis,gan]"
python -m pip install pillow nbformat ipykernel
```

O checkout Pix2Pix também deve estar disponível em `pytorch-CycleGAN-and-pix2pix`
ou no caminho `ABC_PIX2PIX_DIR`. `ABC_DATA_DIR` e `ABC_ARTIFACTS_DIR` configuram dados e artefatos.
O TOML `2324_temporal_gan_value_all_features.toml` fornece os caminhos e os parâmetros; o notebook restringe a execução a
**R5 / Produtividade / rs_puro**. A dose N permanece como metadado para vincular parcelas,
mas não entra como preditor nesse modo.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import json
import os
import shlex
import subprocess
import sys
import tomllib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'pyproject.toml').is_file() and (p / 'src/milho_experiment').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Abra o notebook dentro do checkout do projeto ABC.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from milho_experiment.gan_attributes import (
    GLOBAL_GLCM_CHANNELS, LOCAL_GLCM_CHANNELS, RRENIR_INDEX_NAMES,
    rrenir_attribute_channels,
)
from milho_experiment.runtime import ProjectPaths
from milho_experiment.temporal import (
    AUX_CANDIDATES, HORIZONS, assert_disjoint, build_records, load_record,
    temporal_input, expected_input_channels, paired_stratified_bootstrap_delta,
)
from milho_experiment.pipeline.stage_08_scenarios import stage12_temporal_gan_value as pipeline

PATHS = ProjectPaths.discover(ROOT)
CONFIG_PATH = ROOT / 'configs/stages/08_scenarios/2324_temporal_gan_value_all_features.toml'
CONFIG = tomllib.loads(CONFIG_PATH.read_text(encoding='utf-8'))
MODULE = CONFIG['execution']['module']
DATA, ARTIFACTS = PATHS.data, PATHS.artifacts
STAGE4 = PATHS.expand(CONFIG['inputs']['stage4'])
TABLE = PATHS.expand(CONFIG['inputs']['table'])
REFERENCE_DIR = ARTIFACTS / 'runs/2324_temporal_gan_value/results'

EXECUTAR = False
RECRIAR_RECORTES = False  # opcional; altera os dados de entrada em relação à referência
RUN_ID = 'notebook_temporal_r5_produtividade_all_features_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = ARTIFACTS / 'runs' / RUN_ID
GAN_OUT = OUT / 'results'
RESULT_DIR = REFERENCE_DIR  # troque por GAN_OUT para analisar um novo treino concluído

HORIZON, TARGET, RS_MODE = 'R5', 'Produtividade', 'rs_puro'
BASELINE, CANDIDATE = 'forecast_history', 'forecast_fusion'
CONTRAST = 'forecast_fusion_vs_history'
HISTORY_STAGES = HORIZONS[HORIZON][0]
LABELS = {BASELINE: 'Histórico real', CANDIDATE: 'Histórico real + R5 sintético'}

print('Pipeline:', CONFIG['experiment']['run_id'])
print('Horizonte:', ' + '.join(HISTORY_STAGES), '→', HORIZON)
print('Alvo / modo:', TARGET, '/', RS_MODE)
print('Recortes:', STAGE4, '\nTabela:', TABLE)
print('Resultados consultados:', RESULT_DIR)
print('Nova execução:', GAN_OUT, '\nExecutar processos:', EXECUTAR)


## 2. Fluxo que produz a comparação

```text
Ortomosaicos RRENIR + geometria das parcelas + tabela de campo
    ↓ recortes alinhados e manifesto por parcela/estágio
Histórico real: V6, V8, V13, R2             R5 real: supervisão retrospectiva
    ↓                                            ↓
4 folds externos: 18 parcelas de treino / 6 parcelas de teste
    ↓ seleção de canais auxiliares nos blocos internos
Pix2Pix temporal → R5 sintético fora do treino para cada parcela
    ↓ extração canônica de atributos por imagem
H = atributos de V6 ⊕ V8 ⊕ V13 ⊕ R2       S = atributos do R5 sintético
    ↓                                    ↓
Baseline: regressor(H)         Fusão: regressor(H ⊕ S)
    └────────── produtividade final nas mesmas parcelas ──────────┘
                 ↓ média das predições de 3 sementes
        R² OOF, RMSE, MAE e IC95% pareado de ΔR²
```

`⊕` indica concatenação de atributos. A fusão acrescenta colunas ao vetor de cada parcela;
não aumenta o número de linhas do treino. O R5 real é necessário para treinar e avaliar a GAN
retrospectivamente. **A entrada de previsão de `forecast_history` e `forecast_fusion` não contém
R5 real, biomassa, clorofila de laboratório ou produtividade.**


In [ ]:
requirements = []
for package in ('numpy', 'pandas', 'scipy', 'sklearn', 'PIL', 'torch', 'matplotlib', 'openpyxl'):
    requirements.append({'dependência': package, 'disponível': importlib.util.find_spec(package) is not None})
display(pd.DataFrame(requirements))
display(pd.DataFrame([
    {'entrada': 'manifesto fenológico', 'caminho': str(STAGE4 / 'manifest.csv'),
     'disponível': (STAGE4 / 'manifest.csv').is_file()},
    {'entrada': 'tabela de campo', 'caminho': str(TABLE), 'disponível': TABLE.is_file()},
    {'entrada': 'resultados de referência', 'caminho': str(REFERENCE_DIR),
     'disponível': (REFERENCE_DIR / 'contrasts.csv').is_file()},
]))


## 3. Execução rastreável e preparo opcional dos recortes

O preparo oficial já está disponível no caminho do TOML e é reutilizado por padrão.
O estágio geoespacial recorta Red/RedEdge/NIR, converte a escala numérica e redimensiona para
256 × 256. A implementação atual também rasteriza a máscara poligonal; o manifesto de cada
execução é a fonte para conferir como seus recortes foram produzidos. A normalização por
percentil não substitui calibração radiométrica nem registro espacial entre voos.

Para reconstruir os recortes, habilite `RECRIAR_RECORTES` e `EXECUTAR` e execute a célula
correspondente. Eles serão escritos em `OUT/phenology`; esse caminho passa a ser a entrada do
novo treino. Uma reextração com código atual pode diferir dos recortes legados e deve ser
tratada como uma nova execução, sem esperar reprodução numérica exata da referência.


In [ ]:
def run_module(module, arguments, label):
    arguments = list(map(str, arguments))
    command = [sys.executable, '-m', module, *arguments]
    print(shlex.join(command))
    if not EXECUTAR:
        print('[simulação] Nenhum processo iniciado.')
        return False
    OUT.mkdir(parents=True, exist_ok=True)
    record = {'module': module, 'arguments': arguments, 'python': sys.executable,
              'started_at': datetime.now(timezone.utc).isoformat()}
    record_path = OUT / f'{label}_execution.json'
    record_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    env = os.environ.copy()
    env['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + env.get('PYTHONPATH', '')
    try:
        subprocess.run(command, cwd=ROOT, env=env, check=True)
    except subprocess.CalledProcessError as exc:
        record['exit_code'] = exc.returncode
        raise
    else:
        record['exit_code'] = 0
    finally:
        record['finished_at'] = datetime.now(timezone.utc).isoformat()
        record_path.write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding='utf-8')
    return True


def config_arguments(overrides):
    # Substitui o valor inteiro, inclusive opções TOML com listas de vários elementos.
    args = [str(PATHS.expand(str(a))) if '{' in str(a) else str(a)
            for a in CONFIG['execution']['arguments']]
    for flag, values in overrides.items():
        start = args.index(flag)
        end = start + 1
        while end < len(args) and not args[end].startswith('--'):
            end += 1
        if not isinstance(values, (list, tuple)):
            values = [values]
        args[start + 1:end] = list(map(str, values))
    return args


In [ ]:
if RECRIAR_RECORTES:
    rebuilt_stage4 = OUT / 'phenology'
    rebuilt = run_module('milho_experiment.pipeline.stage_04_phenology.stage4_phenology', [
        '--ortho-dir', DATA / 'raw/safra_2023_2024/orthomosaics',
        '--shapefile', DATA / 'raw/safra_2023_2024/geometry/Shape_parcelas23_24.shp',
        '--out', rebuilt_stage4, '--size', 256, '--reflectance-scale', 'crop',
    ], 'phenology')
    if rebuilt:
        STAGE4 = rebuilt_stage4
else:
    print('Reutilizando os recortes do protocolo:', STAGE4)


## 4. Parcelas completas, produtividade e folds externos

`build_records` exige as cinco imagens da mesma parcela e consulta a tabela por
estágio-alvo/bloco/dose. A implementação também exige biomassa finita em R5 para montar o
registro, embora aqui o alvo previsto seja somente produtividade. Não é feito preenchimento
entre estágios. O protocolo exige **24 parcelas, em quatro blocos de seis**.

Cada teste externo contém um bloco inteiro; os outros três blocos ficam no treino.
As mesmas parcelas e o mesmo alvo são usados nos dois cenários comparados.


In [ ]:
records = None
if (STAGE4 / 'manifest.csv').is_file() and TABLE.is_file():
    records = build_records(STAGE4, TABLE, HORIZON)
    record_frame = pd.DataFrame([
        {'fid': r.fid, 'bloco': r.block, 'dose_n': r.dose_n,
         'histórico': ' + '.join(r.history_stages), 'alvo_imagem': r.target_stage,
         'produtividade': r.productivity}
        for r in records
    ])
    counts = record_frame.groupby('bloco').size()
    assert len(records) == 24 and counts.to_dict() == {1: 6, 2: 6, 3: 6, 4: 6}, counts
    assert record_frame.fid.is_unique
    display(record_frame)
    fold_rows = []
    for block in sorted(counts.index):
        train = [r for r in records if r.block != block]
        test = [r for r in records if r.block == block]
        assert_disjoint(train, test)
        fold_rows.append({'fold externo': block, 'n treino': len(train), 'n teste': len(test),
                          'blocos internos': sorted({r.block for r in train}),
                          'parcelas de teste': [r.fid for r in test]})
    display(pd.DataFrame(fold_rows))
else:
    print('Dados de entrada ausentes: a análise dos CSVs existentes ainda pode ser feita.')


In [ ]:
run_module(MODULE, ['prepare', '--stage4', STAGE4, '--table', TABLE,
                    '--out', OUT / 'prepare', '--horizons', HORIZON], 'prepare_r5')


## 5. Dois níveis de seleção: canais da GAN e atributos do regressor

**A busca considera todos os 38 candidatos já calculáveis com Red/RedEdge/NIR:**

| Grupo | Candidatos | Quantidade |
|---|---|---:|
| Índices | NDVI, NDRE, CIrededge (clorofila), SAVI, EVI2 | 5 |
| GLCM global do NIR | ASM, contraste, correlação, variância, IDM, sum_average, sum_variance, sum_entropy, entropia, diff_entropy, IMC1, IMC2, MCC, dissimilaridade, inércia, cluster_shade, cluster_prominence, homogeneidade | 18 |
| GLCM local do NIR | ASM, contraste, entropia, correlação e homogeneidade, cada uma nas janelas 3, 5 e 7 | 15 |

Os globais têm nomes `glcm_global_<medida>` e usam 16 níveis. Como são escalares por imagem,
entram como planos constantes. A transformação fixa
`0.5 + atan(sign(x) * log1p(abs(x))) / pi` preserva seus valores entre imagens, sem ajustar
limites no teste. Usar percentis de cada plano constante apagaria esse sinal.
Valores globais indefinidos recebem zero bruto (0,5 codificado).
Os locais têm nomes `glcmw3_*`, `glcmw5_*`, `glcmw7_*`, usam 8 níveis e mantêm os mapas espaciais.
Os extratores existentes são reutilizados. A inércia repete o contraste, mas ambos permanecem
no catálogo para contemplar todos os descritores calculados.

CIrededge aparece uma vez; `chlorophyll` continua como alias de compatibilidade.
GNDVI, EVI, VARI e TGI exigem verde/azul, ausentes dos tensores desta pipeline. Eles só podem
entrar em uma versão com essas bandas reais; não são estimados artificialmente.
GLCM de luminância RGB também depende dessas bandas.

**Condicionamento da GAN:** a seleção progressiva começa sem auxiliares e testa todos os
candidatos ainda disponíveis a cada rodada. Mantém-se o limite de **quatro selecionados**;
o conjunto de busca tem 38, não quatro. Red/RedEdge/NIR sempre entram.
Cada auxiliar escolhido é calculado em V6, V8, V13 e R2, na ordem
`[R, RedEdge, NIR, auxiliares]`. A entrada tem **4 × (3 + número de auxiliares)** canais;
a saída contém as três bandas de R5.

A seleção é refeita em cada treino externo, somente nos três blocos internos, com 50 épocas
e semente 7. O escore continua sendo o **ΔR² interno de `combined` contra `forecast_history`**,
conforme `screen_score`; a avaliação principal deste notebook continua sendo `forecast_fusion`.
As escolhas históricas não são fixadas para o novo treino.

**Seleção tabular posterior:** o regressor mantém o extrator e a busca do protocolo anterior:
imputação por mediana e `SelectKBest` (`f_regression`, k = 3, 5 ou 8) dentro da busca interna.
PLSR e Elastic Net incluem padronização. A ampliação solicitada é no conjunto de candidatos
que condicionam a GAN; a definição dos atributos do regressor não foi alterada.


In [ ]:
candidate_catalog = pd.DataFrame({
    'candidato': AUX_CANDIDATES,
    'tipo': [('índice' if n in RRENIR_INDEX_NAMES else 'GLCM global'
              if n in GLOBAL_GLCM_CHANNELS else 'GLCM local') for n in AUX_CANDIDATES],
})
display(candidate_catalog)
print('Candidatos disponíveis:', len(candidate_catalog))
print('Índices:', len(RRENIR_INDEX_NAMES), '| GLCM global:', len(GLOBAL_GLCM_CHANNELS),
      '| GLCM local:', len(LOCAL_GLCM_CHANNELS))
print('Entrada sem auxiliares:', expected_input_channels(HISTORY_STAGES, ()), 'canais')
print('Entrada com quatro auxiliares:', expected_input_channels(HISTORY_STAGES, AUX_CANDIDATES[:4]), 'canais')


## 6. GAN temporal, treinamento final e cross-fitting

- Gerador **U-Net 256**, largura 64 e BatchNorm; discriminador **PatchGAN** (`basic`).
- Perda do gerador: **adversarial vanilla + 100 × L1**; Adam, taxa 0,0002, betas (0,5; 0,999).
- Aumento alinhado entre entrada e alvo: resize para 286, crop aleatório 256 e flip horizontal;
  tensores convertidos de [0, 1] para [−1, 1].
- Treino final: **200 épocas**, batch 4, sementes **7, 11 e 23**.
- Para as 18 parcelas de treino do regressor, as imagens sintéticas são produzidas por
  **cross-fitting**: cada bloco interno é gerado por uma GAN treinada nos outros dois blocos.
- Para as seis parcelas de teste externo, a GAN é treinada nas 18 parcelas restantes.
  O R5 sintético no teste é calculado exclusivamente a partir do histórico dessa parcela.

Na inferência a saída é reconvertida e limitada a [0, 1]. Os checkpoints são reutilizados
quando existem no diretório da nova execução; não misture datasets distintos nesse cache.
O notebook usa um diretório novo para cada sessão de reprodução.


## 7. Atributos agronômicos e baseline comparável

`parcel_features` calcula, por imagem, índices NDVI/NDRE/CIrededge/SAVI, estatísticas espectrais,
número de pixels de vegetação e textura global/local do NIR. A máscara espectral usa NDVI > 0,3;
com menos de 200 pixels, o extrator recorre à imagem inteira. Esses detalhes são mantidos para
refletir o protocolo executado. Não há extensão de GLCM por Red/RedEdge neste experimento.

`H` concatena os atributos das quatro imagens reais; `S` contém os atributos da imagem
R5 sintética. O baseline aprende `H → produtividade`; a fusão aprende `[H, S] → produtividade`.
Ambos têm 18 parcelas no treino e seis no teste em cada fold externo, sem acrescentar dose N.
PLSR, Elastic Net e Extra Trees são comparados por `GridSearchCV` com `GroupKFold` nos blocos
de treino. A família e os hiperparâmetros vencedores podem variar entre folds e sementes.
Não se usa o PLSR histórico de outra tarefa como baseline desta comparação.


In [ ]:
from milho_experiment.pipeline.stage_06_datasets.stage6_features_ortho import parcel_features

feature_names = None
if records is not None:
    history, retrospective_r5 = load_record(records[0])
    feature_names = sorted(parcel_features(history[0]))
    names_h = [f'{stage}__{name}' for stage in HISTORY_STAGES for name in feature_names]
    names_s = [f'R5_sintético__{name}' for name in feature_names]
    display(pd.DataFrame([
        {'cenário': BASELINE, 'entrada do regressor': 'H', 'n atributos antes de SelectKBest': len(names_h)},
        {'cenário': CANDIDATE, 'entrada do regressor': 'H + S',
         'n atributos antes de SelectKBest': len(names_h) + len(names_s)},
    ]))
    display(pd.DataFrame({'atributo por imagem (ordem canônica)': feature_names}))
    print('Tensor base da GAN:', temporal_input(history, ()).shape)
    print('R5 real retrospectivo:', retrospective_r5.shape)
else:
    print('A demonstração das dimensões requer os recortes; os resultados podem ser lidos abaixo.')


## 8. Comando de reprodução do protocolo

A célula usa o novo TOML `2324_temporal_gan_value_all_features.toml`, com
`--candidate-set all` e diretório próprio. Restringe as opções de listas a R5,
Produtividade e `rs_puro`. É mantida a seleção automática de auxiliares, sem `--fixed-attrs`.
O executor canônico também calcula os cenários auxiliares (aumento, sintético sozinho e
combinado); a análise deste notebook filtra apenas `forecast_history` e `forecast_fusion`.

**Execução longa:** com `EXECUTAR=True`, esta célula inicia a triagem, cross-fitting, treinos
finais, busca dos regressores e bootstrap. Os parâmetros não são reduzidos para tornar a
execução mais rápida. Este comando avalia a busca ampliada e **não reproduz o conjunto de candidatos antigo**.
Para reproduzir a referência anterior, use o TOML original, que declara
`--candidate-set legacy` (nove candidatos), com os recortes e parâmetros originais. Hardware e versões numéricas podem introduzir diferenças.


In [ ]:
RUN_ARGUMENTS = config_arguments({
    '--stage4': STAGE4, '--table': TABLE, '--out': GAN_OUT,
    '--horizons': [HORIZON], '--targets': [TARGET], '--rs-modes': [RS_MODE],
})
RUN_SPEC = pipeline.parser().parse_args(RUN_ARGUMENTS)
assert RUN_SPEC.horizons == ['R5'] and RUN_SPEC.targets == ['Produtividade']
assert RUN_SPEC.rs_modes == ['rs_puro'] and RUN_SPEC.fixed_attrs is None
assert RUN_SPEC.candidate_set == 'all' and len(AUX_CANDIDATES) == 38
assert RUN_SPEC.screen_epochs == 50 and RUN_SPEC.final_epochs == 200
assert RUN_SPEC.final_seeds == [7, 11, 23] and RUN_SPEC.max_aux == 4
assert RUN_SPEC.bootstrap == 10_000

display(pd.DataFrame([
    {'parâmetro': name, 'valor': str(getattr(RUN_SPEC, name))}
    for name in ('horizons', 'targets', 'rs_modes', 'screen_epochs', 'screen_seed',
                 'final_epochs', 'final_seeds', 'candidate_set', 'max_aux', 'batch_size', 'bootstrap', 'out')
]))
completed = run_module(MODULE, RUN_ARGUMENTS, 'temporal_r5_produtividade')
if completed:
    RESULT_DIR = GAN_OUT
    print('A análise abaixo usará a nova execução:', RESULT_DIR)


## 9. Carregar a execução e conferir a origem dos números

Por padrão, `RESULT_DIR` aponta para a execução de referência. Após um treino concluído acima,
passa a apontar para `GAN_OUT`. Para consultar outra execução, altere `RESULT_DIR` na próxima célula.
As métricas são lidas dos artefatos da execução, nunca preenchidas com os valores de referência.
Arquivos ausentes geram aviso e a análise dependente é pulada, sem trocar silenciosamente a origem.

O resultado principal é o **R² da média das três predições OOF de cada parcela**.
Ele difere da média dos três R² por semente. Cada parcela precisa ter exatamente uma
predição em cada cenário/semente, com o bloco correspondente mantido fora do treino.


In [ ]:
# Para consultar um treino específico: RESULT_DIR = ARTIFACTS / 'runs/<run-id>/results'
RESULT_DIR = Path(RESULT_DIR)
print('Origem das tabelas:', RESULT_DIR.resolve())

def read_result(name):
    path = RESULT_DIR / name
    if not path.is_file():
        print('Arquivo ausente:', path)
        return None
    return pd.read_csv(path)


def focus(frame, scenarios=None):
    if frame is None:
        return None
    selected = frame[(frame.horizon == HORIZON) & (frame.target == TARGET)]
    if 'rs_mode' in selected:
        selected = selected[selected.rs_mode == RS_MODE]
    if scenarios is not None:
        selected = selected[selected.scenario.isin(scenarios)]
    return selected.copy()

meta_path = RESULT_DIR / 'meta.json'
metadata = json.loads(meta_path.read_text(encoding='utf-8')) if meta_path.is_file() else None
if metadata is not None:
    display(pd.Series(metadata, name='metadados da execução').to_frame())
    assert HORIZON in metadata['horizons'] and TARGET in metadata['targets']
    assert metadata['final_seeds'] == [7, 11, 23]
    actual_candidate_set = metadata.get('candidate_set', 'legacy')
    print('Conjunto usado nesta execução:', actual_candidate_set,
          '| candidatos:', len(metadata.get('candidate_channels', [])))
    if actual_candidate_set != RUN_SPEC.candidate_set:
        print('Referência anterior: estes resultados NÃO avaliam a busca ampliada de 38 candidatos.')
else:
    print('meta.json ausente; a proveniência completa não pôde ser conferida.')

raw_predictions = focus(read_result('oof_predictions.csv'), [BASELINE, CANDIDATE])
ensemble_saved = focus(read_result('oof_predictions_ensemble.csv'), [BASELINE, CANDIDATE])
metrics_saved = focus(read_result('metrics.csv'), [BASELINE, CANDIDATE])
contrasts_all = focus(read_result('contrasts.csv'))
selection = focus(read_result('channel_selection.csv'))
image_metrics = focus(read_result('gan_image_metrics.csv'))


### Canais selecionados e modelos efetivamente utilizados

As escolhas são lidas da execução consultada. Se a origem for a referência antiga,
elas se referem aos nove candidatos anteriores, não à busca ampliada. Uma seleção vazia significa somente as
12 bandas do histórico, sem auxiliares. As escolhas podem mudar entre folds; não é correto
apresentar um único conjunto vencedor como se tivesse sido usado para todas as parcelas.


In [ ]:
selected_channels = None
if selection is not None and not selection.empty:
    final_flag = selection.is_final.astype(str).str.lower().eq('true')
    selected_channels = selection.loc[final_flag, ['outer_block', 'selected', 'delta_r2']].copy()
    selected_channels['selected'] = selected_channels['selected'].fillna('')
    assert len(selected_channels) == 4 and selected_channels.outer_block.is_unique
    selected_channels['n_aux'] = selected_channels.selected.map(lambda s: len(s.split(',')) if s else 0)
    selected_channels['input_nc'] = 4 * (3 + selected_channels.n_aux)
    display(selected_channels.rename(columns={'selected': 'auxiliares', 'delta_r2': 'ΔR² interno combined'}))
if raw_predictions is not None and not raw_predictions.empty:
    model_choices = raw_predictions[['scenario', 'outer_block', 'seed', 'model', 'attrs']].drop_duplicates()
    display(model_choices.sort_values(['scenario', 'outer_block', 'seed']).reset_index(drop=True))


## 10. Recalcular as métricas a partir das predições OOF

Antes da comparação, são conferidos identificadores, blocos, alvos, três sementes e seis parcelas
por fold. As predições são pareadas por parcela e bloco. Os números recalculados precisam
coincidir com `metrics.csv`, `oof_predictions_ensemble.csv` e `contrasts.csv` da mesma execução.
As linhas `rs_dose` e os demais horizontes/alvos não entram nesta conta.


In [ ]:
paired = comparison = contrast_row = None
if raw_predictions is not None and not raw_predictions.empty:
    identity = ['scenario', 'seed', 'fid']
    assert not raw_predictions.duplicated(identity).any(), 'Predição duplicada por parcela/cenário/semente.'
    assert set(raw_predictions.scenario) == {BASELINE, CANDIDATE}
    assert set(raw_predictions.seed) == {7, 11, 23}
    assert len(raw_predictions) == 2 * 3 * 24
    assert raw_predictions[['y', 'prediction']].apply(np.isfinite).all().all()
    assert raw_predictions.groupby('fid').y.nunique().eq(1).all(), 'Alvos inconsistentes.'
    assert raw_predictions.groupby('fid').outer_block.nunique().eq(1).all(), 'Blocos inconsistentes.'
    for _, group in raw_predictions.groupby(['scenario', 'seed']):
        assert len(group) == 24 and group.fid.nunique() == 24
        assert group.groupby('outer_block').size().to_dict() == {1: 6, 2: 6, 3: 6, 4: 6}
    if records is not None:
        current = {r.fid: r for r in records}
        assert set(raw_predictions.fid) == set(current), 'Parcelas diferentes das entradas configuradas.'
        for row in raw_predictions[['fid', 'outer_block', 'y']].drop_duplicates().itertuples():
            assert row.outer_block == current[row.fid].block
            np.testing.assert_allclose(row.y, current[row.fid].productivity)

    ensemble = raw_predictions.groupby(['scenario', 'outer_block', 'fid', 'y'], as_index=False).agg(
        prediction=('prediction', 'mean'), seeds=('seed', 'nunique'))
    assert len(ensemble) == 48 and ensemble.seeds.eq(3).all()
    if ensemble_saved is not None:
        saved = ensemble_saved.set_index(['scenario', 'outer_block', 'fid']).sort_index()
        computed = ensemble.set_index(['scenario', 'outer_block', 'fid']).sort_index()
        assert saved.index.equals(computed.index)
        np.testing.assert_allclose(saved[['y', 'prediction']], computed[['y', 'prediction']], rtol=1e-10)

    paired = ensemble.pivot(index=['outer_block', 'fid', 'y'], columns='scenario',
                            values='prediction').reset_index().sort_values('fid')
    assert len(paired) == 24 and paired[[BASELINE, CANDIDATE]].notna().all().all()
    rows = []
    for scenario in (BASELINE, CANDIDATE):
        rows.append({'scenario': scenario, 'entrada': LABELS[scenario], 'n': len(paired),
                     'r2': r2_score(paired.y, paired[scenario]),
                     'rmse': np.sqrt(mean_squared_error(paired.y, paired[scenario])),
                     'mae': mean_absolute_error(paired.y, paired[scenario])})
    comparison = pd.DataFrame(rows).set_index('scenario')
    if metrics_saved is not None:
        saved = metrics_saved.set_index('scenario').loc[comparison.index]
        np.testing.assert_allclose(comparison[['r2', 'rmse', 'mae']], saved[['r2', 'rmse', 'mae']], rtol=1e-10)
    display(comparison)

    gain = comparison.loc[CANDIDATE, 'r2'] - comparison.loc[BASELINE, 'r2']
    print(f'ΔR² = {gain:+.6f}')
    if contrasts_all is not None:
        selected_contrast = contrasts_all[contrasts_all.contrast == CONTRAST]
        assert len(selected_contrast) == 1, 'Contraste principal ausente ou duplicado.'
        contrast_row = selected_contrast.iloc[0]
        assert contrast_row.baseline == BASELINE and contrast_row.candidate == CANDIDATE
        np.testing.assert_allclose(
            [contrast_row.baseline_r2, contrast_row.candidate_r2, contrast_row.delta_r2],
            [comparison.loc[BASELINE, 'r2'], comparison.loc[CANDIDATE, 'r2'], gain], rtol=1e-10)
        display(selected_contrast.reset_index(drop=True))
else:
    print('Predições do contraste indisponíveis; não há métricas recalculadas para esta origem.')


### Incerteza e estabilidade entre sementes

O IC95% do ganho usa **10.000 reamostragens pareadas de parcelas dentro de cada bloco**,
com semente bootstrap 2026. É o mesmo procedimento do módulo canônico.
Ele reamostra as predições OOF existentes; não retreina toda a seleção em cada amostra bootstrap.
A regra registrada é `ΔR² > 0` e limite inferior do IC95% > 0.
Não há correção por múltiplos contrastes nesse experimento; a comparação foi escolhida entre
resultados já observados. Isso e as 24 parcelas limitam uma interpretação confirmatória do ganho.


In [ ]:
confidence_interval = None
if paired is not None:
    confidence_interval = paired_stratified_bootstrap_delta(
        paired.y, paired[BASELINE], paired[CANDIDATE], paired.outer_block,
        n_bootstrap=RUN_SPEC.bootstrap, seed=2026)
    lo, hi = confidence_interval
    if contrast_row is not None:
        np.testing.assert_allclose([lo, hi], [contrast_row.delta_r2_lo, contrast_row.delta_r2_hi], atol=1e-10)
    supported = bool(gain > 0 and lo > 0)
    print(f'ΔR² {gain:+.6f}; IC95% [{lo:+.6f}; {hi:+.6f}]; regra de melhoria atendida: {supported}')

    seed_rows = []
    for seed, frame in raw_predictions.groupby('seed'):
        scores = {scenario: r2_score(part.y, part.prediction)
                  for scenario, part in frame.groupby('scenario')}
        seed_rows.append({'seed': seed, 'R² histórico': scores[BASELINE],
                          'R² fusão': scores[CANDIDATE], 'ΔR²': scores[CANDIDATE] - scores[BASELINE]})
    seed_comparison = pd.DataFrame(seed_rows)
    display(seed_comparison)
    print('O resultado principal usa a média das predições, e não a média dos R² acima.')


## 11. Visualização da produtividade prevista

Os dois painéis usam exatamente as mesmas parcelas e limites. A reta tracejada representa
predição perfeita; a comparação numérica é feita nas predições OOF agregadas das três sementes.


In [ ]:
if paired is not None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True, layout='constrained')
    limits = [paired[['y', BASELINE, CANDIDATE]].min().min(), paired[['y', BASELINE, CANDIDATE]].max().max()]
    margin = (limits[1] - limits[0]) * 0.06
    limits = [limits[0] - margin, limits[1] + margin]
    for ax, scenario in zip(axes, (BASELINE, CANDIDATE)):
        for block, group in paired.groupby('outer_block'):
            ax.scatter(group.y, group[scenario], label=f'Bloco {block}', s=48)
        ax.plot(limits, limits, '--', color='0.5', linewidth=1)
        ax.set(xlim=limits, ylim=limits, xlabel='Produtividade observada (unidade da tabela)',
               title=f'{LABELS[scenario]}\nR² = {comparison.loc[scenario, "r2"]:.3f}')
        ax.set_aspect('equal', adjustable='box')
        ax.grid(alpha=0.2)
    axes[0].set_ylabel('Produtividade prevista OOF')
    axes[1].legend()
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout='constrained')
    axes[0].bar(['Histórico real', 'Histórico + sintético'], comparison.loc[[BASELINE, CANDIDATE], 'r2'],
                color=['#64748b', '#0f766e'])
    axes[0].set(ylabel='R² OOF', ylim=(min(0, comparison.r2.min() - .05), 1), title='Mesmas 24 parcelas')
    if confidence_interval is not None:
        lo, hi = confidence_interval
        axes[1].plot([lo, hi], [0, 0], color='#0f766e', linewidth=3)
        axes[1].scatter([gain], [0], color='#0f766e', zorder=3)
        axes[1].axvline(0, color='0.5', linestyle='--')
        axes[1].set(xlabel='ΔR²: fusão − histórico', yticks=[], ylim=(-1, 1), title='Ganho e IC95% pareado')
    plt.show()


## 12. Qualidade da imagem gerada e exemplo de inferência

L1, erro de NDVI e erro dos atributos GLCM são diagnósticos da imagem, separados do desempenho
de produtividade. O protocolo não exige que o menor L1 corresponda ao maior R².
Os CSVs guardam métricas, mas não necessariamente as imagens sintéticas.

A prévia opcional restaura **somente um checkpoint externo já existente**, com os canais
escolhidos para aquele fold, e gera a imagem de uma parcela desse bloco de teste.
Ela não treina modelos nem substitui as predições de produtividade agregadas das três sementes.
O R5 real mostrado ao lado serve apenas como referência retrospectiva.


In [ ]:
if image_metrics is not None and not image_metrics.empty:
    assert not image_metrics.duplicated(['fid', 'seed']).any()
    display(image_metrics.groupby('seed')[['l1', 'ndvi_error', 'glcm_feature_mae']].agg(['mean', 'std']))
    print('Diagnósticos referentes somente a R5 / Produtividade na origem:', RESULT_DIR)
else:
    print('Sem métricas de imagem para a execução consultada.')


In [ ]:
GERAR_PREVIA = False  # inferência opcional; requer dados originais e checkpoint externo
PREVIEW_BLOCK, PREVIEW_SEED = 1, 7
if GERAR_PREVIA:
    if records is None or selected_channels is None:
        raise RuntimeError('A prévia exige registros de entrada e seleção de canais disponíveis.')
    if metadata is None or Path(metadata['stage4']).resolve() != STAGE4.resolve():
        raise RuntimeError('Configure STAGE4 com os recortes da execução consultada antes da prévia.')
    choice = selected_channels.set_index('outer_block').loc[PREVIEW_BLOCK, 'selected']
    aux_names = tuple(choice.split(',')) if choice else ()
    checkpoint = pipeline._checkpoint(RESULT_DIR / 'checkpoints/final', HORIZON,
                                      PREVIEW_BLOCK, 'outer', aux_names,
                                      metadata['final_epochs'], PREVIEW_SEED)
    if checkpoint.is_file():
        import torch
        from milho_experiment.pipeline.stage_04_phenology.stage4_phenology import false_color
        device = torch.device('cpu')
        generator = pipeline._new_generator(expected_input_channels(HISTORY_STAGES, aux_names), device)
        payload = torch.load(checkpoint, map_location=device, weights_only=True)
        state = payload['state_dict'] if 'state_dict' in payload else payload
        generator.load_state_dict(state)
        generator.eval()
        test_record = next(r for r in records if r.block == PREVIEW_BLOCK)
        synthetic = pipeline.generate(generator, [test_record], aux_names, device)[test_record.fid]
        real_history, real_r5 = load_record(test_record)
        fig, axes = plt.subplots(1, 3, figsize=(12, 4), layout='constrained')
        for ax, image, title in zip(axes, (real_history[-1], synthetic, real_r5),
                                    ('R2 real (entrada)', 'R5 sintético (saída)', 'R5 real (referência)')):
            ax.imshow(false_color(image))
            ax.set_title(title)
            ax.axis('off')
        fig.suptitle(f'Parcela {test_record.fid} — teste externo do bloco {PREVIEW_BLOCK} — semente {PREVIEW_SEED}')
        plt.show()
        print('Composições falsas-cor com stretch individual; use as métricas para comparação quantitativa.')
    else:
        print('Checkpoint não disponível:', checkpoint)
        print('Os CSVs permitem a análise agronômica, mas a prévia exige restaurar os pesos desta execução.')
else:
    print('Prévia desativada; nenhuma inferência adicional foi iniciada.')


## 13. Relatório do contraste e exportação opcional

O texto abaixo é calculado a partir da origem selecionada e não transforma a referência
em uma meta garantida de novo treino. O ganho representa fusão de atributos com R5 sintético
para previsão antecipada; não demonstra superioridade de sintético puro nem de aumento de dados.
A referência mostra ganho modesto, com limite inferior do intervalo próximo de zero.


In [ ]:
report = None
if comparison is not None and confidence_interval is not None:
    lo, hi = confidence_interval
    statement = ('atende à regra de melhoria registrada' if gain > 0 and lo > 0
                 else 'não atende à regra de melhoria registrada')
    actual_candidate_set = metadata.get('candidate_set', 'legacy') if metadata else 'não identificado'
    report = f'''# Previsão antecipada de produtividade — GAN temporal

Pipeline: `2324_temporal_gan_value`. Conjunto de candidatos: `{actual_candidate_set}`.
Origem: `{RESULT_DIR.resolve()}`. A busca ampliada só foi avaliada se o conjunto for `all`.
Horizonte: V6 + V8 + V13 + R2 → R5. Modo: `{RS_MODE}`, sem dose N como preditor.

- Histórico real: R² OOF = {comparison.loc[BASELINE, 'r2']:.6f}.
- Histórico real + R5 sintético: R² OOF = {comparison.loc[CANDIDATE, 'r2']:.6f}.
- Ganho: ΔR² = {gain:+.6f}; IC95% pareado [{lo:+.6f}; {hi:+.6f}].
- RMSE: {comparison.loc[BASELINE, 'rmse']:.3f} → {comparison.loc[CANDIDATE, 'rmse']:.3f}.
- MAE: {comparison.loc[BASELINE, 'mae']:.3f} → {comparison.loc[CANDIDATE, 'mae']:.3f}.

São 24 parcelas, quatro folds externos por bloco, com média das predições OOF das sementes
7, 11 e 23. O contraste {statement}. O bootstrap usa 10.000 amostras e semente 2026.
A análise é restrita a esta safra e população; a escolha após observar vários contrastes
não recebe correção de multiplicidade nesta avaliação.
'''
    display(Markdown(report))

EXPORTAR_RELATORIO = False
if EXPORTAR_RELATORIO:
    if report is None:
        raise RuntimeError('Carregue e valide as predições antes de exportar.')
    report_dir = OUT / 'report'
    report_dir.mkdir(parents=True, exist_ok=True)
    comparison.to_csv(report_dir / 'comparison.csv')
    paired.to_csv(report_dir / 'paired_oof_predictions.csv', index=False)
    seed_comparison.to_csv(report_dir / 'seed_comparison.csv', index=False)
    (report_dir / 'report.md').write_text(report, encoding='utf-8')
    (report_dir / 'provenance.json').write_text(json.dumps({
        'result_dir': str(RESULT_DIR.resolve()), 'config': str(CONFIG_PATH),
        'horizon': HORIZON, 'target': TARGET, 'rs_mode': RS_MODE,
        'contrast': CONTRAST, 'candidate_set': actual_candidate_set,
        'bootstrap': RUN_SPEC.bootstrap, 'bootstrap_seed': 2026,
    }, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Relatório exportado para:', report_dir)


## Fontes e limites do experimento

- [Configuração da busca completa](configs/stages/08_scenarios/2324_temporal_gan_value_all_features.toml).
- [Configuração histórica com nove candidatos](configs/stages/08_scenarios/2324_temporal_gan_value.toml).
- [Catálogo e codificação dos 38 candidatos](src/milho_experiment/gan_attributes.py).
- [Executor: treino, seleção, cross-fitting, cenários e métricas](src/milho_experiment/pipeline/stage_08_scenarios/stage12_temporal_gan_value.py).
- [Registros, concatenação temporal e bootstrap](src/milho_experiment/temporal.py).
- [Mapas auxiliares canônicos](src/milho_experiment/indices.py).
- [Extrator agronômico usado pelo protocolo](src/milho_experiment/pipeline/stage_06_datasets/stage6_features_ortho.py).
- [Contrastes da execução de referência](artifacts/runs/2324_temporal_gan_value/results/contrasts.csv).

A consulta dos CSVs permite recalcular o ganho registrado; auditar ou reproduzir os treinos
exige também os dados, pesos e ambiente correspondentes. Os resultados de referência não
constituem validação em outra safra. Novas bandas, máscaras, arquiteturas ou perdas mudam o
experimento e devem receber outra execução. Este notebook se concentra no contraste temporal
selecionado, sem incorporar os experimentos de ablação, late fusion ou transferência entre safras.
